# تحلیل مالی چندعاملی - بورس تهران
خروجی فارسی + bypass پراکسی برای scraper

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from crewai import Agent, Task, Crew, Process, LLM

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

llm = LLM(
    model="gpt-4o-mini",
)

## ابزارها - scraper بدون پراکسی

In [8]:
# pip install httpx[socks] requests[socks] socksio
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
import os

class NoProxyScraper(ScrapeWebsiteTool):
    """Scraper که پراکسی رو موقتاً bypass می‌کنه تا سایت‌های داخلی رو بخونه"""
    def _run(self, website_url: str, **kwargs):
        saved_http = os.environ.pop("HTTP_PROXY", None)
        saved_https = os.environ.pop("HTTPS_PROXY", None)
        saved_all = os.environ.pop("ALL_PROXY", None)
        try:
            self.website_url = website_url
            return super()._run(**kwargs)
        finally:
            if saved_http: os.environ["HTTP_PROXY"] = saved_http
            if saved_https: os.environ["HTTPS_PROXY"] = saved_https
            if saved_all: os.environ["ALL_PROXY"] = saved_all

scrape_tool = NoProxyScraper()
search_tool = SerperDevTool()

In [11]:
search_tool._run(query="کلاس ویژن")

{'searchParameters': {'q': 'کلاس ویژن',
  'type': 'search',
  'num': 10,
  'engine': 'google'},
 'organic': [{'title': 'خانه - کلاس\u200cویژن',
   'link': 'https://class.vision/',
   'snippet': 'آموزش بینایی کامپیوتر و یادگیری عمیق. کلاس\u200cویژن، یک سایت تخصصی برای دوره های هوش مصنوعی، دیپ لرنینگ، بینایی کامپیوتر و یادگیری ماشین است.',
   'position': 1},
  {'title': 'کلاس\u200c ویژن',
   'link': 'https://maktabkhooneh.org/organization/%DA%A9%D9%84%D8%A7%D8%B3-%D9%88%DB%8C%DA%98%D9%86-org191/',
   'snippet': 'کلاس\u200cویژن، یک سایت تخصصی برای دوره\u200cهای هوش مصنوعی، دیپ لرنینگ، بینایی کامپیوتر و یادگیری ماشین است. ... سرویس سازمانی مکتب\u200cخونه، بستر رشد و توانمندسازی حرفه\u200cای ...',
   'position': 2},
  {'title': '\u200eکلاس ویژن\u200e (@class.vision) • Instagram photos and videos',
   'link': 'https://www.instagram.com/class.vision/',
   'snippet': 'آموزشهای تخصصی یادگیری عمیق و بینایی کامپیوتر ... پیشرفته\u200cترین ربات انسان نمای جهان به نام آمکا(Ameca) سناریوی ترسناک خود 

In [13]:
scrape_tool._run('https://chartix.ir/')

'The following text is scraped website content:\nنمودار آنلاین بازارهای مالی | نمودار طلا | فارکس، سهام، ارز دیجیتال | چارتیکس\nجستجوی نماد (CTRL+K) خانه بازارها ابزارها تعرفه\u200cها قابلیت\u200cها آموزش بلاگ همکاران ما نمادها لیست تغییرات تماس با ما درباره ما قوانین و مقررات پایگاه دانش بیشتر ورود | ثبت نام ورود جستجوی نماد (CTRL+K) خانه بازارها ابزارها تعرفه\u200cها قابلیت\u200cها آموزش بلاگ همکاران ما نمادها لیست تغییرات تماس با ما درباره ما قوانین و مقررات پایگاه دانش بیشتر نمودار آنلاین چارتیکس معتبرترین مرجع نمودار بازارهای مالی و طلا دسترسی به نمودارهای تکنیکال بازار ایران و جهان با دقت بالا و امکانات پیشرفته نمودارها را تحلیل کنید! ثبت نام (یک روز رایگان) ثبت نام (یک روز رایگان) تاثیرات ما در \xa0اعداد چارتیکس، با ارائه خدمات چارت بازارهای مالی و با بیش از 6 سال تجربه، به\u200cعنوان همراهی مطمئن در دنیای مالی ایران شناخته می\u200cشود سال تجربه بازار فعال + نماد بروکر وتامین کننده / همکار + سال تاریخچه نماد قابلیت های \xa0چارتیکس منابع دریافت دیتا bourse tala forex oanda fxcm f

## تعریف Agentها
نکته مهم: در `goal` هر agent صراحتاً گفته شده **خروجی فارسی**

In [16]:
PERSIAN_INSTRUCTION = "تمام خروجی‌ها، تحلیل‌ها و گزارش‌ها باید کاملاً به زبان فارسی باشند."

data_analyst_agent = Agent(
    role="تحلیلگر داده بازار سرمایه",
    goal=(
        "پایش و تحلیل لحظه‌ای داده‌های بازار بورس تهران "
        "برای شناسایی روندها و پیش‌بینی تحرکات قیمتی. "
        + PERSIAN_INSTRUCTION
    ),
    backstory=(
        "متخصص در بازارهای مالی ایران، از مدل‌سازی آماری "
        "و یادگیری ماشین برای استخراج بینش‌های کلیدی استفاده می‌کند. "
        "ستون فقرات تصمیم‌گیری معاملاتی در تیم است. "
        "همیشه به فارسی گزارش می‌دهد."
    ),
    verbose=True,
    allow_delegation=True,
    llm=llm,
    max_iter=10,
    tools=[scrape_tool, search_tool]
)

In [18]:
trading_strategy_agent = Agent(
    role="توسعه‌دهنده استراتژی معاملاتی",
    goal=(
        "طراحی و آزمون استراتژی‌های معاملاتی "
        "بر اساس تحلیل‌های تحلیلگر داده. "
        + PERSIAN_INSTRUCTION
    ),
    backstory=(
        "با درک عمیق از بازار بورس تهران و تحلیل کمّی، "
        "استراتژی‌های معاملاتی را طراحی و بهینه می‌کند. "
        "رویکردهای مختلف را ارزیابی کرده و سودآورترین "
        "و کم‌ریسک‌ترین گزینه را انتخاب می‌کند. "
        "همیشه به فارسی گزارش می‌دهد."
    ),
    verbose=True,
    allow_delegation=True,
    llm=llm,
    max_iter=10,
    tools=[scrape_tool, search_tool]
)

In [20]:
execution_agent = Agent(
    role="مشاور اجرای معامله",
    goal=(
        "پیشنهاد بهترین روش اجرای معاملات "
        "بر اساس استراتژی‌های تأییدشده. "
        + PERSIAN_INSTRUCTION
    ),
    backstory=(
        "متخصص در تحلیل زمان‌بندی، قیمت و جزئیات لجستیکی معاملات. "
        "با ارزیابی این عوامل، پیشنهادهای مستدلی برای "
        "زمان و نحوه اجرای معامله ارائه می‌دهد. "
        "همیشه به فارسی گزارش می‌دهد."
    ),
    verbose=True,
    allow_delegation=True,
    llm=llm,
    max_iter=10,
    tools=[scrape_tool, search_tool]
)

In [22]:
risk_management_agent = Agent(
    role="مشاور مدیریت ریسک",
    goal=(
        "ارزیابی و ارائه بینش درباره ریسک‌های "
        "مرتبط با فعالیت‌های معاملاتی پیشنهادی. "
        + PERSIAN_INSTRUCTION
    ),
    backstory=(
        "مجهز به درک عمیق از مدل‌های ارزیابی ریسک و دینامیک بازار، "
        "ریسک‌های بالقوه معاملات پیشنهادی را بررسی می‌کند. "
        "تحلیل دقیق ریسک ارائه داده و پیشنهاداتی برای "
        "حفظ انطباق با سطح تحمل ریسک شرکت می‌دهد. "
        "همیشه به فارسی گزارش می‌دهد."
    ),
    verbose=True,
    allow_delegation=True,
    llm=llm,
    max_iter=10,
    tools=[scrape_tool, search_tool]
)

## تعریف Taskها
نکته مهم: در `expected_output` هر task صراحتاً **به فارسی** ذکر شده

In [25]:
data_analysis_task = Task(
    description=(
        "داده‌های بازار بورس تهران را برای نماد ({stock_selection}) "
        "پایش و تحلیل کن. "
        "از مدل‌سازی آماری برای شناسایی روندها "
        "و پیش‌بینی تحرکات قیمتی استفاده کن."
    ),
    expected_output=(
        "گزارش کامل به زبان فارسی شامل: "
        "۱) بینش‌های کلیدی درباره وضعیت فعلی نماد {stock_selection} "
        "۲) روند قیمتی و پیش‌بینی تحرکات آینده "
        "۳) هشدارهای مهم درباره فرصت‌ها یا تهدیدات بازار. "
        "تمام متن باید به فارسی باشد."
    ),
    agent=data_analyst_agent,
)

In [27]:
strategy_development_task = Task(
    description=(
        "بر اساس تحلیل‌های تحلیلگر داده و "
        "سطح تحمل ریسک تعریف‌شده ({risk_tolerance})، "
        "استراتژی‌های معاملاتی را توسعه و اصلاح کن. "
        "رویکرد معاملاتی مورد نظر را هم در نظر بگیر ({trading_strategy_preference})."
    ),
    expected_output=(
        "گزارش کامل به زبان فارسی شامل: "
        "مجموعه‌ای از استراتژی‌های معاملاتی برای نماد {stock_selection} "
        "با جزئیات نقطه ورود، نقطه خروج و حد ضرر. "
        "تمام متن باید به فارسی باشد."
    ),
    agent=trading_strategy_agent,
)

In [29]:
execution_planning_task = Task(
    description=(
        "استراتژی‌های معاملاتی تأییدشده را برای نماد {stock_selection} "
        "تحلیل کن و بهترین روش‌های اجرا را "
        "با توجه به شرایط فعلی بازار و قیمت‌گذاری بهینه مشخص کن."
    ),
    expected_output=(
        "گزارش کامل به زبان فارسی شامل: "
        "برنامه‌های اجرایی دقیق با پیشنهاد زمان‌بندی و نحوه "
        "انجام معاملات برای نماد {stock_selection}. "
        "تمام متن باید به فارسی باشد."
    ),
    agent=execution_agent,
)

In [31]:
risk_assessment_task = Task(
    description=(
        "ریسک‌های مرتبط با استراتژی‌های معاملاتی "
        "و برنامه‌های اجرایی پیشنهادشده برای نماد {stock_selection} را ارزیابی کن. "
        "تحلیل دقیقی از ریسک‌های احتمالی ارائه بده "
        "و استراتژی‌های کاهش ریسک را پیشنهاد کن."
    ),
    expected_output=(
        "گزارش جامع به زبان فارسی شامل: "
        "تحلیل ریسک با جزئیات ریسک‌های احتمالی "
        "و توصیه‌های کاهش ریسک برای نماد {stock_selection}. "
        "تمام متن باید به فارسی باشد."
    ),
    agent=risk_management_agent,
)

## ساخت Crew

In [34]:
financial_trading_crew = Crew(
    agents=[
        data_analyst_agent,
        trading_strategy_agent,
        execution_agent,
        risk_management_agent
    ],
    tasks=[
        data_analysis_task,
        strategy_development_task,
        execution_planning_task,
        risk_assessment_task
    ],
    manager_llm=llm,
    process=Process.hierarchical,
    verbose=True,
    cache=False  # از cache قدیمی استفاده نکن
)

## اجرا

In [37]:
financial_trading_inputs = {
    "stock_selection": "فولاد",
    "initial_capital": "500000000",
    "risk_tolerance": "متوسط",
    "trading_strategy_preference": "نوسان‌گیری کوتاه‌مدت",
    "news_impact_consideration": True
}

In [39]:
# پاک کردن memory قبلی
financial_trading_crew.reset_memories('all')


[2026-05-21 15:36:04][INFO]: [Crew (crew)] Task Output memory has been reset


In [41]:
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.2                                                                                        │
│  Latest version:  1.14.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 91ef3f85-dd56-491e-92fd-c9b7a9466091                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: داده‌های بازار بورس تهران را برای نماد (فولاد) پایش و تحلیل کن. از مدل‌سازی آماری برای شناسایی روندها و    │
│  پیش‌بینی تحرکات قیمتی استفاده کن.                                                                               │
│  ID: d3c1c311-d299-4700-8625-fce7e3ce7d89                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: داده‌های بازار بورس تهران را برای نماد (فولاد) پایش و تحلیل کن. از مدل‌سازی آماری برای شناسایی روندها و    │
│  پیش‌بینی تحرکات قیمتی استفاده کن.                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'پایش و تحلیل داده\u200cهای بازار بورس تهران برای نماد (فولاد) با استفاده از مدل\u200cسازی      │
│  آماری برای شناسایی روندها و پیش\u200cبینی تحرکات قیمتی.', 'context': 'این وظیفه شامل ارائه یک گزارش ...        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#6) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: برای ارزیابی ریسک‌های مرتبط با استراتژی‌های معاملاتی و برنامه‌های اجرایی پیشنهادی برای نماد فولاد، به     │
│  موارد زیر توجه خواهیم کرد:                                                                                     │
│                                                                                                                 │
│  ### ۱. ریسک‌های مرتبط با قیمت‌گذاری فولاد                                                                        │
│  **تحلیل:**                                                                                                     │
│  قیمت‌گذاری فولاد تحت تأثیر عوامل مختلفی قرار دارد که شامل عرضه و تقاضا، هزینه‌های تولید، و سیاست‌های بین‌المللی    │
│  است. افزایش هزینه مواد اولیه یا کاهش تقاضا می‌تواند به شدت بر قیمت فولاد تأثیر بگذارد.                          │
│                                                                                                                 │
│  **استراتژی کاهش ریسک:**                                                                                        │
│  - **تنوع بخشی به سبد سرمایه‌گذاری**: از خرید فولاد به عنوان تنها منبع درآمد خودداری کرده و در سایر نمادها نیز   │
│  سرمایه‌گذاری کنید.                                                                                              │
│  - **استفاده از قراردادهای آتی**: با خرید قراردادهای آتی فولاد می‌توان از نوسانات قیمتی پیشگیری کرد.             │
│                                                                                                                 │
│  ### ۲. نوسانات بازار                                                                                           │
│  **تحلیل:**                                                                                                     │
│  بازار فولاد معمولاً با نوسانات بالا همراه است. عواملی مانند تغییرات اقتصادی، وقوع بحران‌های جهانی، و تغییرات در  │
│  سیاست‌های تجاری ممکن است منجر به نوسانات شدید قیمت‌ها شود.                                                       │
│                                                                                                                 │
│  **استراتژی کاهش ریسک:**                                                                                        │
│  - **تحلیل فنی و بنیادی منظم**: نظارت دقیق بر روندهای پایین و بالا با استفاده از تحلیل‌های بنیادی و فنی به       │
│  پیش‌بینی نوسانات کمک می‌کند.                                                                                     │
│  - **استفاده از ابزارهای مالی محافظتی**: مانند گزینه‌ها و استراتژی‌های هجی، می‌توان برای کاهش ریسک‌های نوسانی       │
│  استفاده کرد.                                                                                                   │
│                                                                                                                 │
│  ### ۳. تغییرات در سیاست‌های اقتصادی                                                                             │
│  **تحلیل:**                                                                                                     │
│  سیاست‌های اقتصادی می‌توانند به عنوان محرک یا موانع عمده بر روی قیمت فولاد عمل کنند. تغییر در تعرفه‌ها، مالیات‌ها   │
│  و مقررات می‌تواند اثرات عمیقی بر صنعت فولاد داشته باشد.                                                         │
│                                                                                                                 │
│  **استراتژی کاهش ریسک:**                                                                                        │
│  - **تحلیل تأثیر سیاست‌ها

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### ارزیابی ریسک‌های مرتبط با استراتژی‌های معاملاتی برای نماد فولاد                                              │
│                                                                                                                 │
│  #### ۱. ریسک‌های مرتبط با قیمت‌گذاری فولاد                                                                       │
│  **تحلیل:**                                                                                                     │
│  قیمت‌گذاری فولاد تحت تأثیر عوامل مختلفی قرار دارد که شامل عرضه و تقاضا، هزینه‌های تولید، و سیاست‌های بین‌المللی    │
│  است. افزایش هزینه مواد اولیه یا کاهش تقاضا می‌تواند به شدت بر قیمت فولاد تأثیر بگذارد.                          │
│                                                                                                                 │
│  **استراتژی کاهش ریسک:**                                                                                        │
│  - **تنوع بخشی به سبد سرمایه‌گذاری**: از خرید فولاد به عنوان تنها منبع درآمد خودداری کرده و در سایر نمادها نیز   │
│  سرمایه‌گذاری کنید.                                                                                              │
│  - **استفاده از قراردادهای آتی**: با خرید قراردادهای آتی فولاد می‌توان از نوسانات قیمتی پیشگیری کرد.             │
│                                                                                                                 │
│  #### ۲. نوسانات بازار                                                                                          │
│  **تحلیل:**                                                                                                     │
│  بازار فولاد معمولاً با نوسانات بالا همراه است. عواملی مانند تغییرات اقتصادی، وقوع بحران‌های جهانی، و تغییرات در  │
│  سیاست‌های تجاری ممکن است منجر به نوسانات شدید قیمت‌ها شود.                                                       │
│                                                                                                                 │
│  **استراتژی کاهش ریسک:**                                                                                        │
│  - **تحلیل فنی و بنیادی منظم**: نظارت دقیق بر روندهای قیمت با استفاده از تحلیل‌های بنیادی و فنی به پیش‌بینی       │
│  نوسانات کمک می‌کند.                                                                                             │
│  - **استفاده از ابزارهای مالی محافظتی**: مانند گزینه‌ها و استراتژی‌های هجی، برای کاهش ریسک‌های نوسانی استفاده      │
│  کرد.                                                                                                           │
│                                                                                                                 │
│  #### ۳. تغییرات در سیاست‌های اقتصادی                                                                            │
│  **تحلیل:**                                                                                                     │
│  سیاست‌های اقتصادی می‌توانند به عنوان محرک یا موانع عمده بر روی قیمت فولاد عمل کنند. تغییر در تعرفه‌ها، مالیات‌ها   │
│  و مقررات می‌تواند اثرات عمیقی بر صنعت فولاد داشته باشد.                                                         │
│                                                                                                                 │
│  **استراتژی کاهش ریسک:**                                                                                        │
│  - **تحلیل تأثیر سیاست‌ها**

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: ریسک‌های مرتبط با استراتژی‌های معاملاتی و برنامه‌های اجرایی پیشنهادشده برای نماد فولاد را ارزیابی کن.       │
│  تحلیل دقیقی از ریسک‌های احتمالی ارائه بده و استراتژی‌های کاهش ریسک را پیشنهاد کن.                                │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 91ef3f85-dd56-491e-92fd-c9b7a9466091                                                                       │
│  Final Output: ### ارزیابی ریسک‌های مرتبط با استراتژی‌های معاملاتی برای نماد فولاد                                │
│                                                                                                                 │
│  #### ۱. ریسک‌های مرتبط با قیمت‌گذاری فولاد                                                                       │
│  **تحلیل:**                                                                                                     │
│  قیمت‌گذاری فولاد تحت تأثیر عوامل مختلفی قرار دارد که شامل عرضه و تقاضا، هزینه‌های تولید، و سیاست‌های بین‌المللی    │
│  است. افزایش هزینه مواد اولیه یا کاهش تقاضا می‌تواند به شدت بر قیمت فولاد تأثیر بگذارد.                          │
│                                                                                                                 │
│  **استراتژی کاهش ریسک:**                                                                                        │
│  - **تنوع بخشی به سبد سرمایه‌گذاری**: از خرید فولاد به عنوان تنها منبع درآمد خودداری کرده و در سایر نمادها نیز   │
│  سرمایه‌گذاری کنید.                                                                                              │
│  - **استفاده از قراردادهای آتی**: با خرید قراردادهای آتی فولاد می‌توان از نوسانات قیمتی پیشگیری کرد.             │
│                                                                                                                 │
│  #### ۲. نوسانات بازار                                                                                          │
│  **تحلیل:**                                                                                                     │
│  بازار فولاد معمولاً با نوسانات بالا همراه است. عواملی مانند تغییرات اقتصادی، وقوع بحران‌های جهانی، و تغییرات در  │
│  سیاست‌های تجاری ممکن است منجر به نوسانات شدید قیمت‌ها شود.                                                       │
│                                                                                                                 │
│  **استراتژی کاهش ریسک:**                                                                                        │
│  - **تحلیل فنی و بنیادی منظم**: نظارت دقیق بر روندهای قیمت با استفاده از تحلیل‌های بنیادی و فنی به پیش‌بینی       │
│  نوسانات کمک می‌کند.                                                                                             │
│  - **استفاده از ابزارهای مالی محافظتی**: مانند گزینه‌ها و استراتژی‌های هجی، برای کاهش ریسک‌های نوسانی استفاده      │
│  کرد.                                                                                                           │
│                                                                                                                 │
│  #### ۳. تغییرات در سیاست‌های اقتصادی                                                                            │
│  **تحلیل:**                                                                                                     │
│  سیاست‌های اقتصادی می‌توانند به عنوان محرک یا موانع عمده بر روی قیمت فولاد عمل کنند. تغییر در تعرفه‌ها، مالیات‌ها   │
│  و مقررات می‌تواند اثرات عمیقی بر صنعت فولاد داشته باشد.                                                         │
│                                                                                                                 │
│  **استراتژی کاهش ریسک:**                                                                                        │
│  - **تحلیل تأثیر سیاست‌ها*

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [43]:
from IPython.display import Markdown

content = result.raw if hasattr(result, "raw") else str(result)
content = content.strip().removeprefix("```markdown").removesuffix("```").strip()
Markdown(content)

### ارزیابی ریسک‌های مرتبط با استراتژی‌های معاملاتی برای نماد فولاد

#### ۱. ریسک‌های مرتبط با قیمت‌گذاری فولاد
**تحلیل:**
قیمت‌گذاری فولاد تحت تأثیر عوامل مختلفی قرار دارد که شامل عرضه و تقاضا، هزینه‌های تولید، و سیاست‌های بین‌المللی است. افزایش هزینه مواد اولیه یا کاهش تقاضا می‌تواند به شدت بر قیمت فولاد تأثیر بگذارد.

**استراتژی کاهش ریسک:**
- **تنوع بخشی به سبد سرمایه‌گذاری**: از خرید فولاد به عنوان تنها منبع درآمد خودداری کرده و در سایر نمادها نیز سرمایه‌گذاری کنید.
- **استفاده از قراردادهای آتی**: با خرید قراردادهای آتی فولاد می‌توان از نوسانات قیمتی پیشگیری کرد.

#### ۲. نوسانات بازار
**تحلیل:**
بازار فولاد معمولاً با نوسانات بالا همراه است. عواملی مانند تغییرات اقتصادی، وقوع بحران‌های جهانی، و تغییرات در سیاست‌های تجاری ممکن است منجر به نوسانات شدید قیمت‌ها شود.

**استراتژی کاهش ریسک:**
- **تحلیل فنی و بنیادی منظم**: نظارت دقیق بر روندهای قیمت با استفاده از تحلیل‌های بنیادی و فنی به پیش‌بینی نوسانات کمک می‌کند.
- **استفاده از ابزارهای مالی محافظتی**: مانند گزینه‌ها و استراتژی‌های هجی، برای کاهش ریسک‌های نوسانی استفاده کرد.

#### ۳. تغییرات در سیاست‌های اقتصادی
**تحلیل:**
سیاست‌های اقتصادی می‌توانند به عنوان محرک یا موانع عمده بر روی قیمت فولاد عمل کنند. تغییر در تعرفه‌ها، مالیات‌ها و مقررات می‌تواند اثرات عمیقی بر صنعت فولاد داشته باشد.

**استراتژی کاهش ریسک:**
- **تحلیل تأثیر سیاست‌ها**: پیش‌بینی چگونگی تأثیر سیاست‌های اقتصادی بر صنعت فولاد و تطبیق استراتژی‌های معاملاتی با شرایط اقتصادی جدید.
- **نظارت بر اخبار و تحولات سیاسی**: رصد مداوم تغییرات سیاسی و اقتصادی به ارزیابی ریسک‌های مرتبط کمک می‌کند.

### جمع‌بندی
برای حفظ انطباق با سطح تحمل ریسک شرکت و محافظت از فرصت‌های سرمایه‌گذاری، پیشنهاد می‌شود که علاوه بر توسعه استراتژی‌های معاملاتی، به تحلیل دقیق عوامل اقتصادی و به‌روز رسانی مداوم به روندهای بازار پرداخته شود. ایجاد یک سیستم هشدار برای نوسانات غیرمنتظره می‌تواند به تصمیم‌گیری بهتر در زمان‌های بحرانی کمک کند.